[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/82_async_retry_solution.ipynb)

# 🟡 Solution: Async Retry Helper

Reference solution for `async_retry`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import asyncio


In [ ]:
# ✅ SOLUTION

async def async_retry(fn, retries: int = 3, delay: float = 0.0,
                      exceptions=(Exception,)):
    last_error = None
    for attempt in range(retries + 1):
        try:
            return await fn()
        except exceptions as e:
            last_error = e
            if attempt == retries:
                raise
            if delay > 0:
                await asyncio.sleep(delay)
    raise last_error


In [ ]:
# Verify

def run_async(coro):
    import asyncio
    import threading
    box = {}
    def target():
        try:
            box['value'] = asyncio.run(coro)
        except BaseException as e:
            box['error'] = e
    t = threading.Thread(target=target)
    t.start()
    t.join(timeout=5)
    if t.is_alive():
        raise TimeoutError('async test timed out')
    if 'error' in box:
        raise box['error']
    return box.get('value')

state = {'n': 0}
async def flaky():
    state['n'] += 1
    if state['n'] < 3:
        raise RuntimeError('try again')
    return 'ok'
print(run_async(async_retry(flaky, retries=3)))


In [ ]:
# Run judge
from torch_judge import check
check('async_retry')
